In [ ]:
# Databricks notebook source
# MAGIC
# MAGIC **Design:**
# MAGIC - Secrets read from Azure Key Vault via Databricks Secret Scope
# MAGIC - Raw JSON landed as-is (append-only, no transformation)
# MAGIC - Partitioned by ingestion date
# MAGIC - Full audit columns on every record

In [ ]:
# Read all secrets from Key Vault via Databricks Secret Scope
ALPHA_VANTAGE_API_KEY = dbutils.secrets.get(scope="retail-banking-scope", key="alpha-vantage-api-key")
SP_CLIENT_ID          = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID          = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET      = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")

# ADLS Gen2 paths
STORAGE_ACCOUNT = "retailbankingdl"
CONTAINER_BRONZE = "bronze"
ADLS_BRONZE_PATH = f"abfss://{CONTAINER_BRONZE}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Alpha Vantage
BASE_URL = "https://www.alphavantage.co/query"

print("✅ Secrets loaded successfully — no values printed for security")

In [ ]:
# Configure Spark to authenticate to ADLS using Service Principal
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print("✅ ADLS Gen2 connection configured via Service Principal")

In [ ]:
import requests
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit, current_timestamp


def fetch_alpha_vantage(params: dict) -> dict:
    """
    Call Alpha Vantage API and return JSON.
    Raises clear errors for API-level failures.
    """
    params["apikey"] = ALPHA_VANTAGE_API_KEY
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "Error Message" in data:
        raise ValueError(f"Alpha Vantage API error: {data['Error Message']}")
    if "Note" in data:
        raise ValueError(f"Alpha Vantage rate limit hit: {data['Note']}")
    if "Information" in data:
        raise ValueError(f"Alpha Vantage limit: {data['Information']}")

    return data


def land_to_bronze(data: dict, source_name: str, table_name: str):
    """
    Land raw JSON as a single record into Bronze Delta table.
    - Append-only (never overwrite raw data)
    - Partitioned by ingestion_date
    - Full audit columns
    """
    ingestion_ts = datetime.utcnow().isoformat()
    ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")

    schema = StructType([
        StructField("raw_json",       StringType(), False),
        StructField("source",         StringType(), False),
        StructField("ingestion_ts",   StringType(), False),
        StructField("ingestion_date", StringType(), False),
    ])

    row = [(
        json.dumps(data),
        source_name,
        ingestion_ts,
        ingestion_date
    )]

    df = spark.createDataFrame(row, schema)

    output_path = f"{ADLS_BRONZE_PATH}/{table_name}"

    (df.write
       .format("delta")
       .mode("append")
       .partitionBy("ingestion_date")
       .save(output_path))

    print(f"✅ {source_name} → {output_path} (partition: {ingestion_date})")
    return output_path

In [ ]:
print("Fetching EUR/USD forex data...")

eurusd_data = fetch_alpha_vantage({
    "function": "FX_DAILY",
    "from_symbol": "EUR",
    "to_symbol": "USD",
    "outputsize": "compact"    # last 100 days
})

land_to_bronze(
    data=eurusd_data,
    source_name="alpha_vantage_forex_eurusd",
    table_name="forex_eurusd"
)

In [ ]:
print("Fetching JPY/USD forex data...")

jpyusd_data = fetch_alpha_vantage({
    "function": "FX_DAILY",
    "from_symbol": "JPY",
    "to_symbol": "USD",
    "outputsize": "compact"
})

land_to_bronze(
    data=jpyusd_data,
    source_name="alpha_vantage_forex_jpyusd",
    table_name="forex_jpyusd"
)

In [ ]:
print("Fetching IBM stock data...")

ibm_data = fetch_alpha_vantage({
    "function": "TIME_SERIES_DAILY",
    "symbol": "IBM",
    "outputsize": "compact"
})

land_to_bronze(
    data=ibm_data,
    source_name="alpha_vantage_stock_ibm",
    table_name="stock_ibm"
)

In [ ]:
print("\n--- Bronze Layer Contents ---")

for table in ["forex_eurusd", "forex_jpyusd", "stock_ibm"]:
    path = f"{ADLS_BRONZE_PATH}/{table}"
    df = spark.read.format("delta").load(path)
    count = df.count()
    print(f"\n📁 {table}: {count} record(s)")
    df.select("source", "ingestion_ts", "ingestion_date").show(truncate=False)

In [ ]:
for table in ["forex_eurusd", "forex_jpyusd", "stock_ibm"]:
    path = f"{ADLS_BRONZE_PATH}/{table}"
    print(f"\n📋 History for {table}:")
    spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
        "version", "timestamp", "operation", "operationParameters"
    ).show(truncate=False)